
# Apache Kafka con Confluent Cloud
## Notebook paso a paso: producir y consumir eventos


En este laboratorio usarás **Kafka como cliente** desde Google Colab para conectarte a un **cluster Kafka gestionado** (Confluent Cloud).

Al final podrás comprender:
- Qué es un **topic** y qué significa que tenga **particiones**
- Qué es una **key** (clave) y cómo influye en la **partición**
- Qué es un **offset** y cómo Kafka recuerda el progreso de lectura
- Qué es un **consumer group** y por qué es importante en streaming

> **Idea clave**: Kafka no es una “librería”. Kafka es un **servicio (broker/cluster)** que vive en Confluent Cloud.  
> En Colab solo instalamos el **cliente** para conectarnos a ese servicio.

---

## Requisitos previos (antes de ejecutar)
En Confluent Cloud ya debes tener:
1. Un **Kafka Cluster** creado (tu “servidor externo”).
2. Un **Topic** creado (por ejemplo `sensores_iot`).
3. Una **API Key / Secret** con permisos para producir/consumir.



## 0) Datos de conexión (OBLIGATORIO)

En esta celda vas a pegar los datos que te da Confluent Cloud.

📍 **Dónde encontrarlos**
- **Bootstrap server**: Cluster → *Cluster settings* → *Bootstrap server*
- **API Key / Secret**: Cluster → *API keys* → crear / copiar valores

✅ Buenas prácticas:
- No publiques estas credenciales en repositorios públicos.
- Usa credenciales dedicadas para laboratorio.


In [ ]:

# ===============================
# 🔐 CONFIGURACIÓN DE CONEXIÓN
# ===============================

BOOTSTRAP_SERVERS = "pkc-921jm.us-east-2.aws.confluent.cloud:9092"  # Ej: pkc-xxxxx.region.provider.confluent.cloud:9092
KAFKA_API_KEY = "YV6TWYJVR6BRI2E7"
KAFKA_API_SECRET = "cfltVtln5Ollhm/YJkHZYl6Woi7NQ8rxY1kbt4v3C6jIQpUFr+KiXzUdVHS7HTdg"

TOPIC_NAME = "sensores_iot"  # Debe existir en tu cluster

print("Configuración cargada. Topic:", TOPIC_NAME)



## 1) Instalar librerías necesarias (cliente Kafka)

Usaremos `confluent-kafka`, un cliente de alto rendimiento.  
También instalamos `pandas` para visualizar eventos en tabla.


In [ ]:

!pip -q install confluent-kafka pandas



## 2) Verificación (cliente): metadata del cluster y particiones del topic

Validamos que:
- podemos conectarnos,
- el cluster responde,
- el topic existe,
- y cuántas particiones tiene.


In [ ]:

from confluent_kafka import Producer

client_conf = {
    "bootstrap.servers": BOOTSTRAP_SERVERS,
    "security.protocol": "SASL_SSL",
    "sasl.mechanisms": "PLAIN",
    "sasl.username": KAFKA_API_KEY,
    "sasl.password": KAFKA_API_SECRET,
}

admin = Producer(client_conf)
metadata = admin.list_topics(timeout=10)

print("✅ Conexión OK. Número de topics visibles:", len(metadata.topics))

if TOPIC_NAME not in metadata.topics:
    raise ValueError(f"⚠️ El topic '{TOPIC_NAME}' NO existe o no tienes permisos. Revisa Confluent Cloud.")
else:
    partitions = len(metadata.topics[TOPIC_NAME].partitions)
    print(f"✔ Topic '{TOPIC_NAME}' existe con {partitions} particiones.")



## 3) Verificación Confluent Cloud Console: comprobar el topic

En Confluent Cloud (otra pestaña):
1) Cluster → **Topics**
2) Abre `sensores_iot`
3) Verifica particiones y retención

✅ Esto confirma que el “servidor Kafka externo” está listo.



## 4) Producir eventos (Producer)

Publicaremos eventos JSON que simulan sensores.

- Usamos `sensor_id` como **key** para que eventos del mismo sensor tiendan a ir a la misma partición.
- Imprimimos `partition` y `offset` para ver que Kafka almacenó el evento.


In [ ]:

import json, random
from datetime import datetime, timezone
from confluent_kafka import Producer

def utc_now():
    return datetime.now(timezone.utc).isoformat()

producer = Producer(client_conf)

def delivery_report(err, msg):
    if err:
        print("❌ Error al enviar:", err)
    else:
        print(f"✅ Enviado | partición={msg.partition()} offset={msg.offset()} key={msg.key().decode() if msg.key() else None}")

N = 20
print(f"Publicando {N} eventos en '{TOPIC_NAME}' ...")

for i in range(N):
    event = {
        "sensor_id": random.randint(1, 4),
        "temp": round(random.uniform(18, 35), 2),
        "timestamp": utc_now(),
        "seq": i
    }
    key = str(event["sensor_id"])
    producer.produce(TOPIC_NAME, key=key, value=json.dumps(event), callback=delivery_report)
    producer.poll(0)

producer.flush()
print("✔ Producción finalizada.")



## 5) Verificación Confluent Cloud Console: ver mensajes producidos

En Confluent Cloud:
1) Cluster → **Topics** → `sensores_iot`
2) Busca **Messages / Browse messages / View messages**
3) Confirma que aparecen eventos recientes

✅ Si ves mensajes, el Producer funcionó.



## 6) Consumir eventos (Consumer)

- `group.id` identifica el grupo de consumidores.
- Kafka guarda offsets por **grupo**, así recuerda el progreso.
- Si el grupo es nuevo, `auto.offset.reset='earliest'` permite leer desde el inicio.


In [ ]:

import time, json
import pandas as pd
from confluent_kafka import Consumer, KafkaException

GROUP_ID = "grupo-1"                       # Puedes dejar este fijo o cambiarlo si quieres re-leer
MAX_MESSAGES = 50                          # Lee hasta 50 mensajes
TOTAL_WAIT_SECONDS = 20                    # O espera máximo 20 segundos

consumer_conf = dict(client_conf)
consumer_conf.update({
    "group.id": GROUP_ID,
    "auto.offset.reset": "earliest",
})

consumer = Consumer(consumer_conf)
consumer.subscribe([TOPIC_NAME])

rows = []
deadline = time.time() + TOTAL_WAIT_SECONDS

print(f"Consumiendo del topic '{TOPIC_NAME}' con group.id='{GROUP_ID}'...")
print(f"Esperaré hasta {TOTAL_WAIT_SECONDS}s o hasta leer {MAX_MESSAGES} mensajes.\n")

try:
    while time.time() < deadline and len(rows) < MAX_MESSAGES:
        msg = consumer.poll(1.0)

        if msg is None:
            # No llegó mensaje en este segundo, seguimos esperando
            continue

        if msg.error():
            raise KafkaException(msg.error())

        rows.append({
            "partition": msg.partition(),
            "offset": msg.offset(),
            "key": msg.key().decode() if msg.key() else None,
            **json.loads(msg.value().decode())
        })
finally:
    consumer.close()

df_events = pd.DataFrame(rows)
print("Eventos recibidos:", len(df_events))
df_events.head(10)




## 7) Experimento: offsets y relectura

1) Ejecuta otra vez el Consumer sin cambiar `GROUP_ID`:
- leerá 0 (o pocos) porque el grupo ya avanzó.

2) Cambia `GROUP_ID` por otro (ej. `grupo-2`) y ejecuta:
- leerá desde el inicio para el nuevo grupo.



## 8) Mini procesamiento (lógica de aplicación)

Kafka transporta eventos; el procesamiento lo hacen consumidores o motores como Flink.

Aquí calculamos:
- promedio por sensor
- alertas si temp ≥ 30


In [ ]:

if df_events.empty:
    print("No hay eventos para procesar.")
else:
    display(df_events.groupby("sensor_id")["temp"].mean().reset_index(name="temp_promedio"))
    THRESHOLD = 30.0
    alerts = df_events[df_events["temp"] >= THRESHOLD].copy()
    print(f"Alertas temp >= {THRESHOLD}: {len(alerts)}")
    display(alerts.sort_values(["timestamp"]).head(10))



## 9) Verificación Confluent Cloud Console: consumer group y consumer lag

En Confluent Cloud, busca:
- **Clients**
- Ingresa a la pestaña **Consumers**
- Ingresa a la pestaña **Consumers lag**
- Verifica que aparezcan los group_id que utilizaste